## load the verlan pairs(290 pairs in total)

In [5]:
import pandas as pd
df = pd.read_csv("../data/processed/verlan_database.csv")
# only takes the words included in the wiktionary_verlan_database.csv
# screen = pd.read_csv("../data/processed/wiktionary_verlan_titles.csv")
# title_set = set(screen["title"].dropna().astype(str).str.strip().str.lower())
# df = df[df["verlan_form"].astype(str).str.strip().str.lower().isin(title_set)].copy()

df_two_cols = df[["verlan_form", "base_form"]]

print(df_two_cols.head(100))
print(f"Rows after screen filter: {len(df)}")

   verlan_form      base_form
0   anti rebeu     antiarabe 
1       babtou         toubab
2   balle-peau  peau de balle
3    ballepeau  peau de balle
4      balpeau  peau de balle
..         ...            ...
95      keubla          black
96        keuf           flic
97        keum            mec
98       keumé            mec
99      keupon           punk

[100 rows x 2 columns]
Rows after screen filter: 290


## jellyfish

In [6]:
import jellyfish
for i, row in df.iterrows():
    w1 = row["verlan_form"]
    w2 = row["base_form"]
    if not isinstance(w1, str) or not isinstance(w2, str):
        print("not valid strings: ", i, w1, type(w1), w2, type(w2))
lev_dist = []
dam_dist = []
jaro_sim = []
normalize_lev_dist = []
for index, row in df_two_cols.iterrows():
    w1 = row["verlan_form"]
    w2 = row["base_form"]
    lev_dist.append(jellyfish.levenshtein_distance(w1, w2))
    dam_dist.append(jellyfish.damerau_levenshtein_distance(w1, w2))
    jaro_sim.append(jellyfish.jaro_similarity(w1, w2))
    normalize_lev_dist.append(jellyfish.levenshtein_distance(w1,w2)/max(len(w1),len(w2)))
    # soundex1 = jellyfish.soundex(w1)
    # soundex2 = jellyfish.soundex(w2)
    # print(soundex1)
    # print(soundex2)
    
print("Levenshtein distance: ", sum(lev_dist) / len(lev_dist))
print("Normalized Levenshtein distance: ", sum(normalize_lev_dist) / len(normalize_lev_dist))
print("Damerau-Levenshtein distance: ", sum(dam_dist) / len(dam_dist))
print("Jaro similarity: ", sum(jaro_sim) / len(jaro_sim))

Levenshtein distance:  4.7620689655172415
Normalized Levenshtein distance:  0.7412203015600582
Damerau-Levenshtein distance:  4.758620689655173
Jaro similarity:  0.4581661544262963


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Build a labeled DataFrame
labels = [f"{row['verlan_form']} → {row['base_form']}" for _, row in df_two_cols.iterrows()]

plot_df = pd.DataFrame({
    "label": labels,
    "verlan": df_two_cols["verlan_form"].values,
    "Levenshtein": lev_dist,
    "Norm. Levenshtein": normalize_lev_dist,
    "Damerau-Levenshtein": dam_dist,
    "Jaro Similarity": jaro_sim,
})

metrics = ["Levenshtein", "Norm. Levenshtein", "Damerau-Levenshtein", "Jaro Similarity"]
palette = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=metrics,
    vertical_spacing=0.14,
    horizontal_spacing=0.10,
)

for idx, (metric, color) in enumerate(zip(metrics, palette)):
    row, col = divmod(idx, 2)
    row += 1
    col += 1

    values = plot_df[metric].values
    # Add small jitter to avoid point overlap
    jitter = np.random.uniform(-0.2, 0.2, size=len(values))

    # Violin plot for background distribution
    fig.add_trace(
        go.Violin(
            y=values,
            name=metric,
            box_visible=True,
            meanline_visible=True,
            fillcolor=color,
            opacity=0.35,
            line_color=color,
            showlegend=False,
            hoverinfo="skip",
            width=0.6,
        ),
        row=row, col=col,
    )

    # Scatter plot with one point per verlan pair
    fig.add_trace(
        go.Scatter(
            x=jitter,
            y=values,
            mode="markers",
            marker=dict(color=color, size=6, opacity=0.75,
                        line=dict(width=0.5, color="white")),
            text=plot_df["label"],
            hovertemplate="<b>%{text}</b><br>" + metric + ": %{y:.3f}<extra></extra>",
            showlegend=False,
        ),
        row=row, col=col,
    )

fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="#eeeeee")

fig.update_layout(
    title=dict(
        text="Verlan ↔ Base Form Distance Distributions (jellyfish)",
        font=dict(size=18),
        x=0.5,
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=13),
    height=750,
    margin=dict(t=90, b=40, l=60, r=40),
)

fig.show()


The distribution of string-based distances shows that French verlan forms are often orthographically far from their standard counterparts. Both raw and normalized Levenshtein distances are generally high, while Jaro similarity is frequently moderate to low. The near-identical distributions of Levenshtein and Damerau-Levenshtein further suggest that verlan is not reducible to simple local character transpositions. Overall, these results indicate that surface-form similarity is insufficient for recovering verlan–base form correspondences, motivating semantic and contextual embedding-based approaches.

## Spacy

In [15]:
import spacy
nlp = spacy.load("fr_core_news_lg")
similarities = []
for index, row in df_two_cols.iterrows():
    w1 = row["verlan_form"]
    w2 = row["base_form"]
    token1 = nlp(w1)
    token2 = nlp(w2)
    sim = token1.similarity(token2)
    similarities.append(sim)
    # print(w1, w2, sim)
print("Average similarity: ", sum(similarities) / len(similarities))
similarities_screen = [sim for sim in similarities if sim > 0]
print("Average similarity (screened): ", sum(similarities_screen) / len(similarities_screen))

/tmp/ipykernel_1990/748199791.py:9: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  sim = token1.similarity(token2)


Average similarity:  0.08325725994173212
Average similarity (screened):  0.2658699440240349


In [16]:
import spacy

nlp = spacy.load("fr_core_news_lg")   # 或 lg
print(nlp.meta["name"])

for w in ["meuf", "femme", "chelou", "louche", "rebeu", "arabe"]:
    doc = nlp(w)
    tok = doc[0]
    print(
        w,
        "has_vector=", tok.has_vector,
        "vector_norm=", tok.vector_norm,
        "is_oov=", tok.is_oov
    )

core_news_lg
meuf has_vector= True vector_norm= 23.746458 is_oov= False
femme has_vector= True vector_norm= 34.48047 is_oov= False
chelou has_vector= True vector_norm= 14.017049 is_oov= False
louche has_vector= True vector_norm= 17.562468 is_oov= False
rebeu has_vector= True vector_norm= 24.750566 is_oov= False
arabe has_vector= True vector_norm= 30.308107 is_oov= False


## Fasttext

In [17]:
import fasttext
import fasttext.util
ft = fasttext.load_model('../artifacts/models/cc.fr.300.bin')



In [18]:
vec = ft.get_word_vector("meuf")
print(vec.shape)
print(vec[:10])

(300,)
[-0.07452467  0.00135089  0.0236647   0.01450148 -0.07200458 -0.000401
  0.16172828  0.03869662 -0.06638871  0.01467166]


In [19]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

w1 = ft.get_word_vector("meuf")
w2 = ft.get_word_vector("femme")

sim = cosine_similarity(w1, w2)
print(sim)
print(ft.get_nearest_neighbors("meuf", k=10))

0.53607875
[(0.7963905930519104, 'nana'), (0.781146228313446, 'gonzesse'), (0.760162889957428, 'pétasse'), (0.7276779413223267, 'mec'), (0.7177290916442871, 'bonnasse'), (0.707476019859314, 'meufs'), (0.7058149576187134, 'pouffiasse'), (0.6935135126113892, 'petasse'), (0.6842332482337952, 'grognasse'), (0.6840963959693909, 'connasse')]


In [20]:
import pandas as pd
import numpy as np

df["fasttext_similarity"] = df.apply(
    lambda row: cosine_similarity(
        ft.get_word_vector(row["verlan_form"]),
        ft.get_word_vector(row["base_form"])
    ),
    axis=1
)
df["fasttext_difference"] = df.apply(
    lambda row: 
        ft.get_word_vector(row["verlan_form"]) - ft.get_word_vector(row["base_form"]),
    axis=1
)



print(df[["verlan_form", "base_form", "fasttext_similarity"]].head(10))
print("Average similarity:", df["fasttext_similarity"].mean())
print("Average difference:", np.linalg.norm(df["fasttext_difference"].mean()))


  verlan_form      base_form  fasttext_similarity
0  anti rebeu     antiarabe              0.333782
1      babtou         toubab             0.476317
2  balle-peau  peau de balle             0.446452
3   ballepeau  peau de balle             0.465137
4     balpeau  peau de balle             0.335395
5       barje         jobard             0.360039
6       barjo         jobard             0.416660
7      barjot         jobard             0.508903
8    barjoter         jobard             0.241804
9       bebar          barbe             0.279475
Average similarity: 0.2619798
Average difference: 0.2625873121399451


In [21]:
import pandas as pd
import numpy as np
from itertools import combinations

def cosine_similarity(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return np.nan
    return np.dot(a, b) / (na * nb)

# 假设 df 有 verlan_form 和 base_form
df = df.dropna(subset=["verlan_form", "base_form"]).copy()
df["verlan_form"] = df["verlan_form"].astype(str).str.strip()
df["base_form"] = df["base_form"].astype(str).str.strip()

# 只保留单词项，先别混入多词表达
df_single = df[
    ~df["verlan_form"].str.contains(r"\s", regex=True) &
    ~df["base_form"].str.contains(r"\s", regex=True)
].copy()

# 计算 shift 向量
shifts = []
labels = []

for _, row in df_single.iterrows():
    v_verlan = ft.get_word_vector(row["verlan_form"])
    v_base = ft.get_word_vector(row["base_form"])
    shift = v_verlan - v_base
    shifts.append(shift)
    labels.append((row["verlan_form"], row["base_form"]))

shifts = np.array(shifts)

# 1. 看每个 shift 的长度
norms = np.linalg.norm(shifts, axis=1)
print("Average shift norm:", norms.mean())
print("Median shift norm:", np.median(norms))

# 2. 看 shift 两两之间的相似度
pair_sims = []
for i, j in combinations(range(len(shifts)), 2):
    sim = cosine_similarity(shifts[i], shifts[j])
    if not np.isnan(sim):
        pair_sims.append(sim)

print("Average pairwise shift similarity:", np.mean(pair_sims))
print("Median pairwise shift similarity:", np.median(pair_sims))

# 3. 看是否存在整体方向
mean_shift = np.mean(shifts, axis=0)

mean_shift_sims = []
for s in shifts:
    sim = cosine_similarity(s, mean_shift)
    if not np.isnan(sim):
        mean_shift_sims.append(sim)

print("Average similarity to mean shift:", np.mean(mean_shift_sims))
print("Median similarity to mean shift:", np.median(mean_shift_sims))

Average shift norm: 1.2289133
Median shift norm: 1.1387718
Average pairwise shift similarity: 0.05164425
Median pairwise shift similarity: 0.04561614
Average similarity to mean shift: 0.23372607
Median similarity to mean shift: 0.2399764


In [10]:
print(ft.get_nearest_neighbors("meuf", k=10))

[(0.7963905930519104, 'nana'), (0.781146228313446, 'gonzesse'), (0.760162889957428, 'pétasse'), (0.7276779413223267, 'mec'), (0.7177290916442871, 'bonnasse'), (0.707476019859314, 'meufs'), (0.7058149576187134, 'pouffiasse'), (0.6935135126113892, 'petasse'), (0.6842332482337952, 'grognasse'), (0.6840963959693909, 'connasse')]


## Camembert

In [24]:
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

input_file = "../outputs/meuf_sentences.txt"
output_file = "../outputs/meuf_femme_similarity.txt"

# 读取句子
with open(input_file, "r", encoding="utf-8") as f:
    sentences_meuf = [line.strip() for line in f if line.strip()]

pattern_meuf = re.compile(r"\bmeufs?\b", re.IGNORECASE)

def replace_meuf(match):
    word = match.group(0)
    if word.lower() == "meufs":
        return "femmes"
    return "femme"

sentences_femme = [pattern_meuf.sub(replace_meuf, s) for s in sentences_meuf]

# 加载 CamemBERT
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
model = AutoModel.from_pretrained("camembert-base")
model.eval()

def sentence_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    # 平均池化
    emb = outputs.last_hidden_state.mean(dim=1).squeeze(0)
    return emb

def cosine_similarity(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

# 计算相似度
results = []
for s_meuf, s_femme in zip(sentences_meuf, sentences_femme):
    emb1 = sentence_embedding(s_meuf)
    emb2 = sentence_embedding(s_femme)
    sim = cosine_similarity(emb1, emb2)
    results.append((s_meuf, s_femme, sim))

# 输出到文件
with open(output_file, "w", encoding="utf-8") as f:
    for s_meuf, s_femme, sim in results:
        f.write(f"ORIGINAL: {s_meuf}\n")
        f.write(f"REPLACED: {s_femme}\n")
        f.write(f"SIMILARITY: {sim:.6f}\n")
        f.write("\n")

# 打印平均值
avg_sim = sum(sim for _, _, sim in results) / len(results)
print("Average similarity:", avg_sim)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2085.20it/s]
CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Average similarity: 0.9720009369186208


In [25]:
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

input_file = "../outputs/meuf_sentences.txt"

# 读取原句
with open(input_file, "r", encoding="utf-8") as f:
    sentences_meuf = [line.strip() for line in f if line.strip()]

pattern_meuf = re.compile(r"\bmeufs?\b", re.IGNORECASE)

def replace_meuf(match):
    word = match.group(0)
    if word.lower() == "meufs":
        return "femmes"
    return "femme"

sentences_femme = [pattern_meuf.sub(replace_meuf, s) for s in sentences_meuf]

# 加载 CamemBERT
tokenizer = AutoTokenizer.from_pretrained("camembert-base", use_fast=True)
model = AutoModel.from_pretrained("camembert-base")
model.eval()

def find_word_span(text, word):
    """
    返回目标词在字符串中的字符区间 [start, end)
    只取第一个完整匹配
    """
    pattern = re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE)
    m = pattern.search(text)
    if m is None:
        return None
    return m.start(), m.end()

def get_target_embedding(text, target_word):
    """
    提取 target_word 在 text 里的 contextual embedding
    做法：找到覆盖该词的所有 subword token，平均它们的向量
    """
    span = find_word_span(text, target_word)
    if span is None:
        return None

    char_start, char_end = span

    encoded = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=128
    )

    offset_mapping = encoded.pop("offset_mapping")[0].tolist()

    with torch.no_grad():
        outputs = model(**encoded)

    # shape: [seq_len, hidden_size]
    hidden = outputs.last_hidden_state[0]

    token_indices = []
    for i, (tok_start, tok_end) in enumerate(offset_mapping):
        # 跳过 special tokens，它们通常 offset 为 (0, 0)
        if tok_start == tok_end == 0:
            continue

        # 只要 token 和目标词字符区间有重叠，就算进去
        if not (tok_end <= char_start or tok_start >= char_end):
            token_indices.append(i)

    if not token_indices:
        return None

    target_vec = hidden[token_indices].mean(dim=0)
    return target_vec

def cosine_similarity(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

results = []

for s_meuf, s_femme in zip(sentences_meuf, sentences_femme):
    vec_meuf = get_target_embedding(s_meuf, "meuf")
    vec_femme = get_target_embedding(s_femme, "femme")

    if vec_meuf is None or vec_femme is None:
        continue

    sim = cosine_similarity(vec_meuf, vec_femme)
    results.append((s_meuf, s_femme, sim))

# 输出结果
for s_meuf, s_femme, sim in results[:10]:
    print("ORIGINAL:", s_meuf)
    print("REPLACED:", s_femme)
    print("TARGET WORD SIMILARITY:", round(sim, 6))
    print()

avg_sim = sum(x[2] for x in results) / len(results)
print("Average target-word similarity:", avg_sim)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2224.39it/s]
CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ORIGINAL: Ça, c'est de la meuf!
REPLACED: Ça, c'est de la femme!
TARGET WORD SIMILARITY: 0.615303

ORIGINAL: Tu vois... j'ai sauvé son petit cul... et c'est lui qui a emballé la meuf.
REPLACED: Tu vois... j'ai sauvé son petit cul... et c'est lui qui a emballé la femme.
TARGET WORD SIMILARITY: 0.62326

ORIGINAL: Ouais, c'est un peu comme être avec une meuf sans dents.
REPLACED: Ouais, c'est un peu comme être avec une femme sans dents.
TARGET WORD SIMILARITY: 0.642098

ORIGINAL: Sans meuf, on baise pas.
REPLACED: Sans femme, on baise pas.
TARGET WORD SIMILARITY: 0.58248

ORIGINAL: La meuf a reçu un verre.
REPLACED: La femme a reçu un verre.
TARGET WORD SIMILARITY: 0.658817

ORIGINAL: Cette meuf te kife, mec.
REPLACED: Cette femme te kife, mec.
TARGET WORD SIMILARITY: 0.664354

ORIGINAL: Je vais me taper une meuf avant le mariage.
REPLACED: Je vais me taper une femme avant le mariage.
TARGET WORD SIMILARITY: 0.695051

ORIGINAL: Cette meuf, Stephanie, y a rien à jeter chez elle.
REPLACED: 

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# 批量处理多个 verlan -> base_form
pairs_df = (
    df_two_cols.dropna(subset=["verlan_form", "base_form"])
    .assign(
        verlan_form=lambda x: x["verlan_form"].astype(str).str.strip(),
        base_form=lambda x: x["base_form"].astype(str).str.strip(),
    )
    .query("verlan_form != '' and base_form != ''")
    .drop_duplicates(subset=["verlan_form", "base_form"])
    .reset_index(drop=True)
)

tokenizer = AutoTokenizer.from_pretrained("camembert-base", use_fast=True)
model = AutoModel.from_pretrained("camembert-base")
model.eval()

templates = [
    "Ce mot est courant: {}.",
    "J'entends souvent {} dans la rue.",
    "Dans cette phrase, le mot est {}.",
]

def batched_sentence_embeddings(texts, batch_size=32, max_length=64):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            encoded = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            )
            outputs = model(**encoded)
            hidden = outputs.last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1)
            masked_hidden = hidden * mask
            pooled = masked_hidden.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            all_embs.append(pooled)
    return torch.cat(all_embs, dim=0)

verlan_texts = [
    tpl.format(v)
    for v in pairs_df["verlan_form"].tolist()
    for tpl in templates
]
base_texts = [
    tpl.format(b)
    for b in pairs_df["base_form"].tolist()
    for tpl in templates
]

verlan_embs = batched_sentence_embeddings(verlan_texts)
base_embs = batched_sentence_embeddings(base_texts)

n_pairs = len(pairs_df)
n_tpl = len(templates)
verlan_embs = verlan_embs.view(n_pairs, n_tpl, -1)
base_embs = base_embs.view(n_pairs, n_tpl, -1)

pair_sims = F.cosine_similarity(verlan_embs, base_embs, dim=-1).mean(dim=1)

batch_result = pairs_df.copy()
batch_result["camembert_similarity"] = pair_sims.cpu().numpy()
batch_result = batch_result.sort_values("camembert_similarity", ascending=False)

print(batch_result.head(20))
print("Average similarity:", batch_result["camembert_similarity"].mean())

/home/wan/miniconda3/envs/llm-verlan/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'df_two_cols' is not defined